In [1]:
from column import TNN_Col
from dendrite import ActiveDendrite
from submodule import *
from backend.backend import * 
import argparse
import os
from rich.console import Console
from tnn_mdls.func_mdls import *
from tnn_mdls.tb_func_mdls import *
import time
from veriloggen import *
import copy

In [2]:
layers = []

In [3]:
temp, _ = ActiveDendrite(num_col=2, num_neurons=2, num_dend=1, p_dist=4, p_prox=1, num_seg=1, wres_dist=3, wres_prox=3, thres=6).TNN_Layer()
layers.append(temp)

In [4]:
temp, _ = ActiveDendrite(num_col=1, num_neurons=1, num_dend=1, p_dist=4, p_prox=1, num_seg=1, wres_dist=3, wres_prox=3, thres=6).TNN_Layer()
layers.append(temp)

In [5]:
layers

In [6]:
model = Module('model')
clk = model.Input('clk')
grst = model.Input('grst')
rstb = model.Input('rstb')

In [7]:
# Generate model (wrapper) ports
for i in range(len(layers)):
    layer = layers[i]
    ports = copy.deepcopy(layer.get_ports())
    # model input ports should match layer 1 input ports
    if (i==0):
        # iterate through ports
        for key in ports:
            port = ports[key]
            port_name = port.name
            # skip clk, grst, rstb
            if ((port_name!='clk') & (port_name!='grst') & (port_name!='rstb')):
                # add input ports to model
                if (isinstance(port, core.vtypes.Input)):
                    port.name = 'L0_'+port.name
                    model.add_object(port)
    # add non-connecting input ports to model for other layers
    else:
        # iterate through ports
        for key in ports:
            port = ports[key]
            port_name = port.name
            # skip clk, grst, rstb
            if ((port_name!='clk') & (port_name!='grst') & (port_name!='rstb')):
                # add input ports to model
                if (isinstance(port, core.vtypes.Input)):
                    # filter out connecting ports
                    if (not(port_name.startswith('input_spikes_dist'))):
                        port.name = 'L'+str(i)+'_'+port.name
                        model.add_object(port)
                        
        # generate output port
        if (i == len(layers)-1):
            for key in ports:
                port = ports[key]
                port_name = port.name
                if (isinstance(port, core.vtypes.Output)):
                    model.add_object(port)

In [8]:
model.get_ports()

OrderedDict([('clk', <veriloggen.core.vtypes.Input at 0x1d815c153f0>),
             ('grst', <veriloggen.core.vtypes.Input at 0x1d815c15240>),
             ('rstb', <veriloggen.core.vtypes.Input at 0x1d817cb2140>),
             ('L0_input_spikes_dist_0_0',
              <veriloggen.core.vtypes.Input at 0x1d817da10f0>),
             ('L0_input_spikes_dist_0_1',
              <veriloggen.core.vtypes.Input at 0x1d817da1b10>),
             ('L0_input_spikes_dist_1_0',
              <veriloggen.core.vtypes.Input at 0x1d817da30d0>),
             ('L0_input_spikes_dist_1_1',
              <veriloggen.core.vtypes.Input at 0x1d817da3130>),
             ('L0_input_spikes_prox_0',
              <veriloggen.core.vtypes.Input at 0x1d817da2230>),
             ('L0_w_init_dist_0_0_000',
              <veriloggen.core.vtypes.Input at 0x1d817da22c0>),
             ('L0_w_init_dist_0_0_001',
              <veriloggen.core.vtypes.Input at 0x1d817da23b0>),
             ('L0_w_init_dist_0_0_002',
         

In [9]:
model_ports = model.get_ports()

for i in range(len(layers)):
    layer = layers[i]
    layer_ports = [clk, grst, rstb]
    
    # For first layer, add all ports with prefix L0
    if i == 0:
        for key in model_ports:
            if (key).startswith('L'+str(i)):
                layer_ports.append(model_ports[key])
                
        # Instantiate wire to connect output to next layer's input
        neuron_count = layers[i].get_params()['NUM_NEURONS'].value * layers[i].get_params()['NUM_COL'].value
        last_out = model.Wire('out_'+str(i)+'_in_'+str(i+1), neuron_count)
        layer_ports.append(last_out)
    else:
        # distal input is last layer's output
        layer_ports.extend(last_out)
        
        for key in model_ports:
            if (key).startswith('L'+str(i)):
                layer_ports.append(model_ports[key])
                
        # Instantiate wire to connect output to next layer's input
        last_out = []
        for c in range(layers[i].get_params()['NUM_COL'].value):
            last_out.append(model.Wire('out_'+str(i)+'_in_'+str(i+1)+'_'+str(c), layers[i].get_params()['NUM_NEURONS'].value))
        layer_ports.extend(last_out)

    model.Instance(layer, 'L'+str(i)+'_'+layer.name, params=layer.get_params(),
                  ports = layer_ports)

In [10]:
gen_file, rtl_path = gen_verilog(module = model, filename = 'model.v')